# A2 Q3 - Build the NRMS baseline inputs (run locally, upload to Kaggle)

Stages everything `src/nrms_baseline_kaggle.ipynb` needs into
`data/kaggle_nrms/`, one set of files per dataset:

| file | contents |
|---|---|
| `nrms_{dataset}_train.parquet` | seeded sample of `train` impressions, for fitting |
| `nrms_{dataset}_val.parquet` | **the same** 200,000 val impressions A2 Q2 evaluated on |
| `nrms_{dataset}_test.parquet` | **the same** 200,000 test impressions A2 Q2 evaluated on |
| `nrms_{dataset}_history.parquet` | per-user click history, truncated to the last 20 |
| `{dataset}_article_embeddings.parquet` | A1 Q3's document vectors (`NRMSDocVec`'s news input) |
| `nrms_inputs_manifest.json` | row counts, sampling parameters, recency-weight basis |

Why a local prep step at all: `behaviors.parquet` is 24.6M rows / 7.8GB for
`ebnerd_large`, so the sampling has to happen before anything is uploaded.
The evaluation population is drawn by the *same function* as
`reranker_evaluation.ipynb` at the same seed, so Q3's NRMS numbers land on
byte-identical impressions to Q2's BM25 / embedding / re-ranker numbers and
the two tables can be read side by side. When
`reranker_eval_{split}.parquet` already exists the test cell asserts that
identity directly rather than trusting the shared seed.

In [1]:
import json
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path

import polars as pl

from cs4406m26_assignment1c1.retrieval import RECENT_N_CLICKS


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
OUT_DIR = ROOT / "data" / "kaggle_nrms"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_LOG = ROOT / "build_progress.log"

# The history window every stage of this project uses. Held at A1's
# RECENT_N_CLICKS so NRMS reads exactly the clicks BM25, the embedding
# retriever and the re-ranker read; it is also ebrec's own
# hparams_nrms_docvec.history_size default, so the baseline runs at its
# published setting and the comparison holds the input window constant.
HISTORY_SIZE = 20

# Identical to reranker_evaluation.ipynb's constants, on purpose: same
# sampling function, same seed, therefore the same impressions.
EVAL_IMPRESSIONS = 200_000
EVAL_SEED = 0

# Training sample. 400,000 impressions matches the cap A2 Q2 measured a
# learning curve for (SPEC.md A2 Q2 #9), which keeps the two Stage-2 models
# trained on comparable volumes of the same split.
TRAIN_IMPRESSIONS = 400_000
TRAIN_SEED = 0

SPLITS = ["val", "test"]

# Set False if A1's {dataset}_article_embeddings.parquet is already attached
# to the Kaggle notebook (e.g. as compute_embeddings_kaggle.ipynb's own
# output); the Kaggle notebook discovers them anywhere under /kaggle/input.
COPY_EMBEDDINGS = True

_env_datasets = os.environ.get("NRMS_INPUT_DATASETS")
DATASETS = _env_datasets.split(",") if _env_datasets else ["ebnerd_large", "mind_large"]


def log_progress(message: str) -> None:
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] nrms_inputs: {message}\n")
        f.flush()


def write_parquet_atomic(df: pl.DataFrame, path: Path) -> Path:
    """Write-then-rename, the same discipline as every other persisted
    artifact here: a direct write leaves a truncated file behind if the
    process dies mid-write, and every caller treats existence as
    completeness."""
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.write_parquet(tmp)
    os.replace(tmp, path)
    return path


log_progress(f"nrms_inputs started (datasets={DATASETS})")

paths = {
    name: {
        "behaviors": DATA_DIR / name / "behaviors.parquet",
        "history": DATA_DIR / name / "history.parquet",
        "embeddings": DATA_DIR / name / "article_embeddings.parquet",
    }
    for name in DATASETS
}
{name: {k: v.exists() for k, v in p.items()} for name, p in paths.items()}

{'ebnerd_large': {'behaviors': True, 'history': True, 'embeddings': True}}

In [2]:
def test_setup() -> None:
    for name in DATASETS:
        for key, path in paths[name].items():
            assert path.exists(), f"{name}: missing {key} at {path}"
    # NRMS must see the same history window as every other method, or the
    # "before vs after" comparison is confounded by the input, not the model.
    assert HISTORY_SIZE == RECENT_N_CLICKS, (HISTORY_SIZE, RECENT_N_CLICKS)
    # The evaluation population is only shared with A2 Q2 if the constants
    # match; read them back from Q2's own output rather than assuming.
    for name in DATASETS:
        q2_metrics = DATA_DIR / name / "reranker_eval_metrics.json"
        if not q2_metrics.exists():
            continue
        hp = json.loads(q2_metrics.read_text(encoding="utf-8"))["hyperparameters"]
        assert hp["eval_impressions"] == EVAL_IMPRESSIONS, (name, hp["eval_impressions"])
        assert hp["eval_seed"] == EVAL_SEED, (name, hp["eval_seed"])
        assert hp["recent_n_clicks"] == HISTORY_SIZE, (name, hp["recent_n_clicks"])


test_setup()
print("setup OK:", {name: str(paths[name]["behaviors"].parent) for name in DATASETS})

setup OK: {'ebnerd_large': 'C:\\Users\\HP\\Desktop\\Coursework\\IRE\\cs4406m26-assignment1c1\\data\\processed\\ebnerd_large'}


## Evaluation population - the same impressions A2 Q2 used

`sample_eval_impressions` is `reranker_evaluation.ipynb`'s function
unchanged, including the detail that made it reproducible in the first
place: the impression ids are sorted *before* sampling, because polars'
`unique`/scan order gives no ordering guarantee and sampling straight off
it drew a different subset per run even at a fixed seed (SPEC.md A2 Q2 #4).

In [3]:
def sample_eval_impressions(dataset: str, split: str) -> pl.DataFrame:
    lf = pl.scan_parquet(paths[dataset]["behaviors"]).filter(pl.col("split") == split)
    ids = lf.select("impression_id").collect()["impression_id"].sort()
    n = min(EVAL_IMPRESSIONS, ids.len())
    if n < ids.len():
        ids = ids.sample(n=n, seed=EVAL_SEED)
    return (
        lf.filter(pl.col("impression_id").is_in(ids.implode()))
        .select("impression_id", "user_id", "impression_time", "article_ids_inview", "article_ids_clicked")
        .collect()
        .sort(["user_id", "impression_id"])
    )


eval_population = {}
for name in DATASETS:
    for split in SPLITS:
        df = sample_eval_impressions(name, split)
        eval_population[(name, split)] = df
        log_progress(f"  {name}/{split}: {df.height:,} eval impressions sampled")
{f"{n}/{s}": eval_population[(n, s)].height for n, s in eval_population}

{'ebnerd_large/val': 200000, 'ebnerd_large/test': 200000}

In [4]:
def test_eval_population() -> None:
    for name in DATASETS:
        for split in SPLITS:
            df = eval_population[(name, split)]
            available = (
                pl.scan_parquet(paths[name]["behaviors"])
                .filter(pl.col("split") == split)
                .select(pl.len())
                .collect()
                .item()
            )
            assert df.height == min(EVAL_IMPRESSIONS, available), (name, split, df.height, available)
            assert df["impression_id"].n_unique() == df.height, f"{name}/{split}: duplicate impressions"
            # Every impression must be scorable and gradeable: an empty inview
            # set has nothing to rank, and AUC/MRR/nDCG need at least one click.
            assert df["article_ids_inview"].list.len().min() > 0
            assert df["article_ids_clicked"].list.len().min() > 0
            # The clicked ids have to be inside the inview set, or every
            # metric is silently capped (SPEC.md A2 Q2 #11).
            outside = (
                df.select(
                    pl.col("article_ids_clicked")
                    .list.set_difference(pl.col("article_ids_inview"))
                    .list.len()
                    .sum()
                ).item()
            )
            assert outside == 0, f"{name}/{split}: {outside} clicked ids outside their inview set"
            # Same population as A2 Q2, checked rather than inferred from the
            # shared seed, whenever Q2's own output is on disk.
            q2_path = DATA_DIR / name / f"reranker_eval_{split}.parquet"
            if q2_path.exists():
                q2_ids = (
                    pl.scan_parquet(q2_path).select("impression_id").unique().collect()["impression_id"].sort()
                )
                ours = df["impression_id"].sort()
                assert q2_ids.len() == ours.len(), (name, split, q2_ids.len(), ours.len())
                assert q2_ids.to_list() == ours.to_list(), f"{name}/{split}: population differs from A2 Q2's"
        val_ids = set(eval_population[(name, "val")]["impression_id"].to_list())
        test_ids = set(eval_population[(name, "test")]["impression_id"].to_list())
        assert not (val_ids & test_ids), f"{name}: val and test populations overlap"


test_eval_population()
print("evaluation population OK (reproducible, disjoint, gradeable, identical to A2 Q2's where comparable)")

evaluation population OK (reproducible, disjoint, gradeable, identical to A2 Q2's where comparable)


## Training sample

Same sort-then-sample discipline as above. `train` is A1's own split (every
impression strictly before the val cutoff, SPEC.md Q1 #3), so it cannot
overlap the evaluation population by construction - the test cell checks it
anyway, since a silent overlap would inflate every number downstream.

The last calendar day of this sample becomes NRMS's early-stopping
validation set inside the Kaggle notebook, mirroring
`ebnerd_nrms_docvec.py`'s own `last_dt` split. That keeps our `val` split
untouched by model selection: it is reported, not tuned against.

In [5]:
def sample_train_impressions(dataset: str) -> pl.DataFrame:
    lf = pl.scan_parquet(paths[dataset]["behaviors"]).filter(pl.col("split") == "train")
    ids = lf.select("impression_id").collect()["impression_id"].sort()
    n = min(TRAIN_IMPRESSIONS, ids.len())
    if n < ids.len():
        ids = ids.sample(n=n, seed=TRAIN_SEED)
    return (
        lf.filter(pl.col("impression_id").is_in(ids.implode()))
        .select("impression_id", "user_id", "impression_time", "article_ids_inview", "article_ids_clicked")
        .collect()
        .sort(["user_id", "impression_id"])
    )


train_population = {}
for name in DATASETS:
    df = sample_train_impressions(name)
    train_population[name] = df
    log_progress(f"  {name}/train: {df.height:,} train impressions sampled")
{name: train_population[name].height for name in DATASETS}

{'ebnerd_large': 400000}

In [6]:
def test_train_population() -> None:
    for name in DATASETS:
        df = train_population[name]
        available = (
            pl.scan_parquet(paths[name]["behaviors"])
            .filter(pl.col("split") == "train")
            .select(pl.len())
            .collect()
            .item()
        )
        assert df.height == min(TRAIN_IMPRESSIONS, available), (name, df.height, available)
        assert df["impression_id"].n_unique() == df.height
        assert df["article_ids_clicked"].list.len().min() > 0, f"{name}: an impression with no click"
        # Negative sampling needs at least one non-clicked candidate to draw
        # from; wu2019 sampling with replacement tolerates a short pool but
        # not an empty one.
        n_no_negatives = (
            df.select(
                (
                    pl.col("article_ids_inview").list.len()
                    - pl.col("article_ids_clicked").list.unique().list.len()
                    <= 0
                ).sum()
            ).item()
        )
        print(f"   {name}: {n_no_negatives:,} impressions with no negative candidate")
        train_ids = set(df["impression_id"].to_list())
        for split in SPLITS:
            overlap = train_ids & set(eval_population[(name, split)]["impression_id"].to_list())
            assert not overlap, f"{name}: train sample overlaps {split} population ({len(overlap)} ids)"
        pos_rate = (
            df.select(
                pl.col("article_ids_clicked").list.unique().list.len().sum()
                / pl.col("article_ids_inview").list.len().sum()
            ).item()
        )
        print(f"   {name}: train positive rate {pos_rate:.4f}, mean inview {df['article_ids_inview'].list.len().mean():.2f}")


test_train_population()
print("train population OK (reproducible, disjoint from val/test, every impression has a positive)")

   ebnerd_large: 0 impressions with no negative candidate


   ebnerd_large: train positive rate 0.0902, mean inview 11.12
train population OK (reproducible, disjoint from val/test, every impression has a positive)


## History, truncated to the last 20 clicks

Only the users these three impression files actually reference are kept -
`ebnerd_large`'s `history.parquet` holds 974,791 users, of which the
sampled populations touch a fraction, and the uploaded file is otherwise
dominated by users nothing will ever look up.

`timestamp_sequence` is carried **only** for datasets that really have it,
and the capability is decided by null counts rather than column dtype. The
two MIND tracks write the same semantically-absent column with different
types (`mind` as `Null`, `mind_large` as `String`, both 100% null), so a
`dtype != pl.Null` test classified `mind_large` as having real timestamps
and would derive elapsed-time recency weights from timestamps that do not
exist (A2 Q1 #3). MIND's recency basis is therefore the ordinal proxy, and
the manifest records which basis each dataset got.

In [7]:
def has_timestamps(dataset: str) -> bool:
    """True only if history.timestamp_sequence carries at least one real
    value. Null-count based, never dtype based - see the cell above."""
    non_null = (
        pl.scan_parquet(paths[dataset]["history"])
        .select(pl.col("timestamp_sequence").is_not_null().sum())
        .collect()
        .item()
    )
    return bool(non_null and non_null > 0)


def build_history(dataset: str, needed_users: pl.Series) -> pl.DataFrame:
    columns = ["user_id", "article_id_sequence"]
    keep_timestamps = has_timestamps(dataset)
    if keep_timestamps:
        columns.append("timestamp_sequence")
    lf = (
        pl.scan_parquet(paths[dataset]["history"])
        .select(columns)
        .filter(pl.col("user_id").is_in(needed_users.implode()))
        .with_columns(pl.col("article_id_sequence").list.tail(HISTORY_SIZE))
    )
    if keep_timestamps:
        lf = lf.with_columns(pl.col("timestamp_sequence").list.tail(HISTORY_SIZE))
    return lf.collect().sort("user_id")


history = {}
recency_basis = {}
for name in DATASETS:
    users = (
        pl.concat(
            [train_population[name].select("user_id")]
            + [eval_population[(name, split)].select("user_id") for split in SPLITS]
        )["user_id"]
        .unique()
        .sort()
    )
    history[name] = build_history(name, users)
    recency_basis[name] = "elapsed_time" if has_timestamps(name) else "ordinal_proxy"
    log_progress(
        f"  {name}: history rows {history[name].height:,} for {users.len():,} referenced users "
        f"(basis={recency_basis[name]})"
    )
{name: {"rows": history[name].height, "basis": recency_basis[name]} for name in DATASETS}

{'ebnerd_large': {'rows': 396290, 'basis': 'elapsed_time'}}

In [8]:
def test_history() -> None:
    for name in DATASETS:
        hist = history[name]
        assert hist["user_id"].n_unique() == hist.height, f"{name}: duplicate history rows"
        assert hist["article_id_sequence"].list.len().max() <= HISTORY_SIZE
        # Every impression's user must have a row, including cold-start users,
        # whose row is an empty list rather than an absent row (A1 Q1 #7) -
        # a missing row would make NRMS silently skip those impressions.
        referenced = (
            pl.concat(
                [train_population[name].select("user_id")]
                + [eval_population[(name, split)].select("user_id") for split in SPLITS]
            )["user_id"]
            .unique()
        )
        missing = set(referenced.to_list()) - set(hist["user_id"].to_list())
        assert not missing, f"{name}: {len(missing)} referenced users have no history row"
        n_coldstart = int((hist["article_id_sequence"].list.len() == 0).sum())
        print(f"   {name}: {n_coldstart:,} cold-start users of {hist.height:,}, basis {recency_basis[name]}")
        if recency_basis[name] == "elapsed_time":
            assert "timestamp_sequence" in hist.columns
            # Truncation has to keep the two sequences aligned, or every
            # recency weight is attached to the wrong click.
            mismatched = (
                hist.select(
                    (
                        pl.col("article_id_sequence").list.len() != pl.col("timestamp_sequence").list.len()
                    ).sum()
                ).item()
            )
            assert mismatched == 0, f"{name}: {mismatched} rows with misaligned history sequences"
            # Ascending order is what makes "the last N" mean "the most
            # recent N" (A1 Q2 #4); checked on a sample rather than on
            # ~800k rows of nested lists.
            sample = hist.filter(pl.col("timestamp_sequence").list.len() > 1).head(2000)
            # Cast to Int64 (microseconds) rather than differencing datetimes:
            # a Duration compared against a literal 0 is a type error, and
            # the cast is exact.
            non_monotonic = (
                sample.select(
                    pl.col("timestamp_sequence")
                    .list.eval((pl.element().cast(pl.Int64).diff() < 0).any())
                    .list.first()
                    .sum()
                ).item()
            )
            assert non_monotonic == 0, f"{name}: {non_monotonic} histories are not in ascending time order"
        else:
            assert "timestamp_sequence" not in hist.columns, (
                f"{name}: has no timestamp data but a timestamp column was written"
            )


test_history()
print("history OK (one row per referenced user, truncated, aligned, ascending)")

   ebnerd_large: 0 cold-start users of 396,290, basis elapsed_time
history OK (one row per referenced user, truncated, aligned, ascending)


## Write the upload set

In [9]:
written = {}
for name in DATASETS:
    written[name] = {
        "train": write_parquet_atomic(train_population[name], OUT_DIR / f"nrms_{name}_train.parquet"),
        "val": write_parquet_atomic(eval_population[(name, "val")], OUT_DIR / f"nrms_{name}_val.parquet"),
        "test": write_parquet_atomic(eval_population[(name, "test")], OUT_DIR / f"nrms_{name}_test.parquet"),
        "history": write_parquet_atomic(history[name], OUT_DIR / f"nrms_{name}_history.parquet"),
    }
    if COPY_EMBEDDINGS:
        target = OUT_DIR / f"{name}_article_embeddings.parquet"
        shutil.copyfile(paths[name]["embeddings"], target)
        written[name]["embeddings"] = target
    log_progress(f"  {name}: upload set written to {OUT_DIR}")

total_mb = sum(p.stat().st_size for d in written.values() for p in d.values()) / 1e6
{
    "upload_dir": str(OUT_DIR),
    "total_mb": round(total_mb, 1),
    "files": {name: {k: round(p.stat().st_size / 1e6, 1) for k, p in d.items()} for name, d in written.items()},
}

{'upload_dir': 'C:\\Users\\HP\\Desktop\\Coursework\\IRE\\cs4406m26-assignment1c1\\data\\kaggle_nrms',
 'total_mb': 681.8,
 'files': {'ebnerd_large': {'train': 17.8,
   'val': 8.3,
   'test': 9.3,
   'history': 53.5,
   'embeddings': 592.9}}}

In [10]:
def test_written() -> None:
    for name in DATASETS:
        for key, path in written[name].items():
            assert path.exists() and path.stat().st_size > 0, f"{name}: {key} not written"
            assert not path.with_suffix(path.suffix + ".tmp").exists(), f"{name}: {key} left a .tmp behind"
        # Round-trip every impression file: the Kaggle notebook reads exactly
        # these bytes, so a schema surprise has to fail here, not there.
        for split in ["train", "val", "test"]:
            back = pl.read_parquet(written[name][split])
            expected = train_population[name] if split == "train" else eval_population[(name, split)]
            assert back.height == expected.height, (name, split, back.height, expected.height)
            assert back.columns == [
                "impression_id",
                "user_id",
                "impression_time",
                "article_ids_inview",
                "article_ids_clicked",
            ], (name, split, back.columns)
            assert back["impression_time"].dtype == pl.Datetime, (name, split, back["impression_time"].dtype)
        back_hist = pl.read_parquet(written[name]["history"])
        assert back_hist.height == history[name].height
        # Every article id the Kaggle notebook will look up must have an
        # embedding, or NRMS scores a zero vector for it without complaining.
        emb_ids = pl.read_parquet(paths[name]["embeddings"], columns=["article_id"])["article_id"]
        emb_set = set(emb_ids.to_list())
        inview_ids = pl.concat(
            [
                pl.read_parquet(written[name][split], columns=["article_ids_inview"])
                for split in ["train", "val", "test"]
            ]
        )["article_ids_inview"]
        referenced = set(inview_ids.explode().drop_nulls().unique().to_list()) | set(
            back_hist["article_id_sequence"].explode().drop_nulls().unique().to_list()
        )
        uncovered = referenced - emb_set
        coverage = 1.0 - len(uncovered) / max(1, len(referenced))
        print(f"   {name}: {len(referenced):,} referenced articles, embedding coverage {coverage:.6f}")
        assert coverage == 1.0, f"{name}: {len(uncovered)} referenced articles have no embedding"


test_written()
print("upload set OK (round-trips, no .tmp left behind, every referenced article has an embedding)")

C:\Users\HP\AppData\Local\Temp\ipykernel_18356\2535263743.py:32: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  referenced = set(inview_ids.explode().drop_nulls().unique().to_list()) | set(


C:\Users\HP\AppData\Local\Temp\ipykernel_18356\2535263743.py:33: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  back_hist["article_id_sequence"].explode().drop_nulls().unique().to_list()


   ebnerd_large: 19,112 referenced articles, embedding coverage 1.000000
upload set OK (round-trips, no .tmp left behind, every referenced article has an embedding)


## Manifest

In [11]:
manifest = {
    "schema_version": 1,
    "build_timestamp": datetime.now(timezone.utc).isoformat(),
    "hyperparameters": {
        "history_size": HISTORY_SIZE,
        "eval_impressions": EVAL_IMPRESSIONS,
        "eval_seed": EVAL_SEED,
        "train_impressions": TRAIN_IMPRESSIONS,
        "train_seed": TRAIN_SEED,
    },
    "datasets": {
        name: {
            "recency_weight_basis": recency_basis[name],
            "has_timestamps": recency_basis[name] == "elapsed_time",
            "impressions": {
                "train": train_population[name].height,
                "val": eval_population[(name, "val")].height,
                "test": eval_population[(name, "test")].height,
            },
            "history_rows": history[name].height,
            "coldstart_users": int((history[name]["article_id_sequence"].list.len() == 0).sum()),
            "embeddings_copied": bool(COPY_EMBEDDINGS),
        }
        for name in DATASETS
    },
}

manifest_path = OUT_DIR / "nrms_inputs_manifest.json"
# Merge rather than overwrite: a run scoped to one dataset via
# NRMS_INPUT_DATASETS must not delete the other dataset's entry, since both
# are uploaded together.
if manifest_path.exists():
    previous = json.loads(manifest_path.read_text(encoding="utf-8"))
    merged = dict(previous.get("datasets", {}))
    merged.update(manifest["datasets"])
    manifest["datasets"] = merged
tmp = manifest_path.with_suffix(".json.tmp")
tmp.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
os.replace(tmp, manifest_path)
log_progress(f"  manifest written ({sorted(manifest['datasets'])})")
manifest

{'schema_version': 1,
 'build_timestamp': '2026-09-11T14:50:22.257393+00:00',
 'hyperparameters': {'history_size': 20,
  'eval_impressions': 200000,
  'eval_seed': 0,
  'train_impressions': 400000,
  'train_seed': 0},
 'datasets': {'mind_large': {'recency_weight_basis': 'ordinal_proxy',
   'has_timestamps': False,
   'impressions': {'train': 400000, 'val': 200000, 'test': 200000},
   'history_rows': 445145,
   'coldstart_users': 10077,
   'embeddings_copied': True},
  'ebnerd_large': {'recency_weight_basis': 'elapsed_time',
   'has_timestamps': True,
   'impressions': {'train': 400000, 'val': 200000, 'test': 200000},
   'history_rows': 396290,
   'coldstart_users': 0,
   'embeddings_copied': True}}}

In [12]:
def test_manifest() -> None:
    back = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert back["hyperparameters"]["history_size"] == HISTORY_SIZE
    for name in DATASETS:
        entry = back["datasets"][name]
        assert entry["recency_weight_basis"] in {"elapsed_time", "ordinal_proxy"}
        assert entry["impressions"]["val"] == eval_population[(name, "val")].height
        assert entry["impressions"]["test"] == eval_population[(name, "test")].height
        assert entry["impressions"]["train"] == train_population[name].height
        assert entry["history_rows"] == history[name].height
        # MIND has no click timestamps at all, ever (A1 Q1 #2); EB-NeRD does.
        assert entry["has_timestamps"] == (entry["recency_weight_basis"] == "elapsed_time")


test_manifest()
log_progress("nrms_inputs finished")
print("manifest OK")

manifest OK


# Manual Verification Complete

Upload everything in `data/kaggle_nrms/` as one private Kaggle Dataset
(Kaggle -> New Notebook -> **+ Add Data -> Upload -> New Dataset**), then
import `src/nrms_baseline_kaggle.ipynb`, set **Accelerator -> GPU** and
**Internet -> On**, and Save & Run All.

`{dataset}_article_embeddings.parquet` is only staged here when
`COPY_EMBEDDINGS` is True; if A1's embeddings are already attached to the
Kaggle notebook from `compute_embeddings_kaggle.ipynb`'s output, set it
False and attach that dataset instead - the Kaggle notebook discovers the
files anywhere under `/kaggle/input`.